In [2]:
!pip install fastembed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 36.5 MB/s eta 0:00:00


In [3]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

dense_model = SentenceTransformer("all-MiniLM-L6-v2")
sparse_model = SparseTextEmbedding("Qdrant/bm25")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

In [21]:
!pip install qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 25.9 MB/s eta 0:00:00


In [22]:
from qdrant_client import QdrantClient


client = QdrantClient(
    url=url,
    api_key=api_key)

In [4]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_json("hf://datasets/Amod/mental_health_counseling_conversations/combined_dataset.json", lines=True)

In [8]:
df.head(1)['Context'].values

array(["I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?"],
      dtype=object)

In [31]:
query = df.head(1)['Context'].values

In [33]:
type(str(query))

str

In [36]:
query = str(df.head(1)['Context'].values)

query_dense  = dense_model.encode(query).tolist()
query_sparse = list(sparse_model.embed(query))[0]

In [37]:
from qdrant_client.models import SparseVector,Prefetch,FusionQuery,Fusion

results = client.query_points(
    collection_name="VectorDatabase",
    prefetch=[
        Prefetch(query=query_dense, using="dense", limit=10),
        Prefetch(
            query=SparseVector(
                indices=query_sparse.indices.tolist(),
                values=query_sparse.values.tolist(),
            ),
            using="sparse",
            limit=10,
        ),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=5,
    with_payload=True,
)

In [40]:
type(results)

qdrant_client.http.models.models.QueryResponse

In [42]:
type(results.points)

list

In [43]:
len(results.points)

5

In [45]:
type(results.points[0])

qdrant_client.http.models.models.ScoredPoint

In [51]:
for i in range(0,5):
  print(results.points[i].score)

0.7
0.64285713
0.45
0.44444445
0.39285713


In [54]:
results.points[0].payload['Context']

"I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?"

In [59]:
results.points[0].payload['Response']

'Hello, and thank you for your question and seeking advice on this. Feelings of worthlessness is unfortunately common. In fact, most people, if not all, have felt this to some degree at some point in their life. You are not alone.\xa0Changing our feelings is like changing our thoughts - it\'s hard to do. Our minds are so amazing that the minute you change your thought another one can be right there to take it\'s place. Without your permission, another thought can just pop in there. The new thought may feel worse than the last one! My guess is that you have tried several things to improve this on your own even before reaching out on here. People often try thinking positive thoughts, debating with their thoughts, or simply telling themselves that they need to "snap out of it" - which is also a thought that carries some self-criticism.\xa0Some people try a different approach, and there are counselors out there that can help you with this. The idea is that instead of trying to change the t

In [67]:
def CheckQuery(query):
  if isinstance(query, str):
      return True

  try:
      return str(query)
  except :
      return False

def retreive(query):
  if not CheckQuery(query):
    return -1

  query_dense  = dense_model.encode(query).tolist()
  query_sparse = list(sparse_model.embed([query]))[0]

  results = client.query_points(
      collection_name="VectorDatabase",
      prefetch=[
          Prefetch(query=query_dense, using="dense", limit=10),
          Prefetch(
              query=SparseVector(
                  indices=query_sparse.indices.tolist(),
                  values=query_sparse.values.tolist(),
              ),
              using="sparse",
              limit=10,
          ),
      ],
      query=FusionQuery(fusion=Fusion.RRF),
      limit=5,
      with_payload=True,
  )

  results_list = []
  for i in range(5):
    results_list.append((results.points[i].payload['Context'],results.points[i].payload['Response']))

  return results_list


In [71]:
retreive(query)

[("I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?",
  'Hello, and thank you for your question and seeking advice on this. Feelings of worthlessness is unfortunately common. In fact, most people, if not all, have felt this to some degree at some point in their life. You are not alone.\xa0Changing our feelings is like changing our thoughts - it\'s hard to do. Our minds are so amazing that the minute you change your thought another one can be right there to take it\'s place. Without your permission, another thought can just pop in there. The new thought may feel worse than the last one! My guess is that you have tried several things to improve this on your own even before reaching out on here. People often try